# 01. Environment and population structure

This notebook reads the analysis-ready 31-variable environmental matrix, including the prepared Striga HS variables, characterizes population and environmental structure, and produces Figure 1.


# 01. 环境与群体结构

本 notebook 读取包含已准备 Striga HS 变量的 31-variable analysis-ready environmental matrix，刻画群体与环境结构，并生成 Figure 1。


## Remote kernel initialization

The notebook runs against the remote project configuration. Verify the reported host, working directory, Python executable, environment, and expected kernelspec before continuing. This cell does not download data or start an analysis.


## 远程 kernel 初始化

本 notebook 根据远程项目配置运行。继续之前，请核对输出的 host、工作目录、Python executable、environment 和预期 kernelspec。本单元不会下载数据或启动分析。


In [ ]:
import os
import socket
import sys
from pathlib import Path

import yaml

%load_ext autoreload
%autoreload 2

launch_dir = Path.cwd().resolve()
project_file = launch_dir / "config" / "project.yaml"
if not project_file.is_file() and launch_dir.name == "notebooks":
    project_file = launch_dir.parent / "config" / "project.yaml"
if not project_file.is_file():
    raise FileNotFoundError("Copy config/project.example.yaml to config/project.yaml and complete it before running this notebook")
with project_file.open("r", encoding="utf-8") as stream:
    project_settings = yaml.safe_load(stream)
PROJECT_ROOT = Path(project_settings["remote"]["project_root"]).expanduser().resolve()
allowed_launch_dirs = {PROJECT_ROOT, PROJECT_ROOT / "notebooks"}
assert launch_dir in allowed_launch_dirs, f"Start Jupyter from the configured remote project root: {PROJECT_ROOT}"
assert PROJECT_ROOT.is_dir(), f"Remote project does not exist: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
expected_environment = project_settings["jupyter"]["kernel"]["environment"]["name"]
active_environment = os.environ.get("CONDA_DEFAULT_ENV", Path(sys.prefix).name)
if project_settings["jupyter"]["same_environment"]:
    server_environment = project_settings["jupyter"]["server"]["environment"]["name"]
    assert server_environment == expected_environment, (server_environment, expected_environment)
assert active_environment == expected_environment, (active_environment, expected_environment)
print({
    "host": socket.gethostname(),
    "working_directory": str(Path.cwd()),
    "python_executable": sys.executable,
    "environment": active_environment,
    "expected_kernelspec": project_settings["jupyter"]["kernel"]["kernelspec"],
})


## Run mode selection

Use `checkpoint` to read the published intermediate inputs or `full` to rebuild this notebook's intermediate products from source data. Outputs from the current run are always written to generated directories and never overwrite reference files.


## 运行模式选择

使用 `checkpoint` 读取已发布的中间输入；使用 `full` 从 source data 重建本 notebook 的中间产品。本次运行的输出始终写入 generated 目录，不覆盖 reference 文件。


In [ ]:
RUN_MODE = "checkpoint"
if RUN_MODE not in {"checkpoint", "full"}:
    raise ValueError("RUN_MODE must be checkpoint or full")
RUN_MODE


## Inputs and output contracts

All scientific inputs are resolved below `PROJECT_ROOT/data`. Published inputs are read from source or reference directories; newly built checkpoints and results are written only to generated directories. The following sections validate the notebook-specific schemas before interpreting results.


## 输入与输出契约

所有科学输入均从 `PROJECT_ROOT/data` 下解析。已发布输入从 source 或 reference 目录读取；新建 checkpoints 和 results 只写入 generated 目录。以下章节在解释结果前检查本 notebook 特有的 schemas。


## Scientific settings used in this notebook

The expected accession count, environmental-variable count, and focal gradients are analysis choices used below. They are kept with the scientific record instead of the technical project configuration.


## 本 notebook 使用的科学设置

预期样本数、环境变量数和重点环境梯度是以下分析使用的科学选择，因此保留在研究记录中，而不是技术项目配置中。


In [ ]:
EXPECTED_LANDRACE_COUNT = 443
EXPECTED_ENVIRONMENTAL_VARIABLE_COUNT = 31
FOCAL_VARIABLES = ["gsl", "bio18", "al", "striga"]


In [ ]:
import sys
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from laenrich.config import load_project_paths
from laenrich.data import read_environment_inputs, source_path
from laenrich.environment import environmental_correlation_network, fit_rda, zscore_frame
from laenrich.io import validate_analysis_mode
from laenrich.plotting import save_figure
paths = load_project_paths(PROJECT_ROOT / "config" / "project.yaml")
mode = validate_analysis_mode(RUN_MODE)
checkpoint_dir = paths.checkpoint_reference_dir / "environment" if mode == "checkpoint" else paths.checkpoint_generated_dir / "environment"
figure_data_dir = paths.result_generated_dir / "figure_data" / "figure1"
figure_dir = paths.result_generated_dir / "figures"
supplementary_table_dir = paths.result_generated_dir / "supplementary_tables"
supplementary_dataset_dir = paths.result_generated_dir / "supplementary_datasets"
audit_dir = paths.result_generated_dir / "audits" / "01_environment"
for directory in [figure_data_dir, figure_dir, supplementary_table_dir, supplementary_dataset_dir, audit_dir]:
    directory.mkdir(parents=True, exist_ok=True)
print({"mode": mode, "source_dir": str(paths.source_dir), "checkpoint_dir": str(checkpoint_dir)})


## 1. Landrace metadata and public environmental variables

The accession table must contain the 443 GEA landraces, canonical identifiers, WGS84 coordinates, botanical type, and geographic metadata. Environmental columns are matched to the versioned 31-variable catalog before transformation.


## 1. Landrace metadata 与公开环境变量

样本表必须包含 443 个 GEA landraces、canonical identifiers、WGS84 坐标、botanical type 和地理元数据。环境列在转换前与版本化的 31-variable catalog 对齐。


In [ ]:
landraces, variable_catalog = read_environment_inputs(paths.source_dir)
environment_columns = variable_catalog["env_id"].tolist()
landrace_indexed = landraces.set_index("LIB", verify_integrity=True)
environment_raw = landrace_indexed.loc[:, environment_columns].apply(pd.to_numeric, errors="coerce")
assert environment_raw.shape == (443, 31)
missingness = environment_raw.isna().mean().rename("missing_fraction").to_frame()
complete_mask = environment_raw.notna().all(axis=1)
environment_standardized = zscore_frame(environment_raw.loc[complete_mask])
assert environment_raw.shape[1] == int(EXPECTED_ENVIRONMENTAL_VARIABLE_COUNT)
missingness.sort_values("missing_fraction", ascending=False).head(10)


## 2. Provenance of the Striga variables

striga and striga_crop are analysis-ready variables in landraces_metadata.csv. Their upstream occurrence filtering, MaxEnt model selection, and distance rule are documented in the manuscript but are outside this repository's reproducibility entry point. This notebook does not reconstruct occurrence or raster products.


## 2. Striga 变量的来源

striga 和 striga_crop 是 landraces_metadata.csv 中的 analysis-ready variables。其上游 occurrence filtering、MaxEnt model selection 和 distance rule 由论文记录，但不属于本仓库的复现起点。本 notebook 不重建 occurrence 或 raster products。


In [ ]:
striga_provenance = pd.DataFrame({
    "variable_id": ["striga", "striga_crop"],
    "analysis_label": ["Striga HS", "Striga HS (Crop)"],
    "entry_point": ["landraces_metadata.csv", "landraces_metadata.csv"],
    "status": ["analysis-ready source", "analysis-ready source"],
})
striga_provenance


## 3. Striga HS and Striga HS (Crop)

The two prepared scores are selected directly from the landrace metadata and renamed only in exported analysis tables. No additional zero assignment or spatial transformation is applied here.


## 3. Striga HS 与 Striga HS (Crop)

这两个准备好的 scores 直接从 landrace metadata 中选择，仅在导出的分析表中重命名；这里不再进行额外的 zero assignment 或空间转换。


In [ ]:
striga_scores = environment_raw[["striga", "striga_crop"]].rename(
    columns={"striga": "striga_hs", "striga_crop": "striga_hs_crop"}
)
striga_scores.describe()


## 4. Final environmental matrix

The final matrix follows the catalog order. Missing values and transformations are reported explicitly, and the complete matrix is standardized for RDA while raw-scale values remain available for interpretation and envGWAS.


## 4. 最终环境矩阵

最终矩阵严格遵循 catalog 顺序。缺失值和转换被明确报告；完整矩阵经标准化后用于 RDA，同时保留原尺度数据供解释和 envGWAS 使用。


In [ ]:
environment_raw = landrace_indexed[environment_columns].astype(float)
missingness = environment_raw.isna().mean().rename("missing_fraction").to_frame()
complete_mask = environment_raw.notna().all(axis=1)
environment_standardized = zscore_frame(environment_raw.loc[complete_mask])
assert environment_raw.shape[1] == int(EXPECTED_ENVIRONMENTAL_VARIABLE_COUNT)
missingness.sort_values("missing_fraction", ascending=False).head(10)


## 5. Population structure, environmental correlations, and RDA

The neighbor-joining tree is built externally from LD-pruned SNPs. Here, 987,993 unlinked SNPs are used for RDA, environmental correlations are summarized at |r| > 0.7, and constrained variance is checked against the manuscript value.


## 5. 群体结构、环境相关性与 RDA

Neighbor-joining tree 在外部由 LD-pruned SNPs 构建。本节使用 987,993 个 unlinked SNPs 进行 RDA，以 |r| > 0.7 汇总环境相关性，并将 constrained variance 与稿件数值核对。


In [ ]:
genotypes = pd.read_hdf(checkpoint_dir / "population_structure.h5", key="genotypes")
if "canonical_accession_id" in genotypes:
    genotypes = genotypes.set_index("canonical_accession_id")
genotypes = genotypes.loc[environment_standardized.index]
rda = fit_rda(genotypes, environment_standardized)
nodes, edges = environmental_correlation_network(environment_raw, threshold=0.7)
rda_summary = pd.DataFrame({
    "metric": ["explained_variance", "adjusted_r_squared", "rda1_variance", "rda2_variance"],
    "value": [rda.explained_variance, rda.adjusted_r_squared,
              rda.constrained_variance_ratio[0], rda.constrained_variance_ratio[1]],
})
rda_summary


## 6. Four focal environmental gradients

GSL and PHQ represent complementary drought gradients; AL represents soil aluminum toxicity; and STR represents Striga pressure. All 31 variables remain in RDA and envGWAS, while these four guide detailed downstream analyses.


## 6. 四个重点环境梯度

GSL 和 PHQ 表示互补的干旱梯度，AL 表示土壤铝毒，STR 表示 Striga pressure。31 个变量全部保留用于 RDA 和 envGWAS，以下四个变量用于详细下游分析。


In [ ]:
focal_variables = FOCAL_VARIABLES
focal_catalog = variable_catalog.set_index("env_id").loc[focal_variables]
focal_catalog[["env_abbr", "short description", "source"]]


## 7. Figure 1 and exported datasets

Figure 1 combines four environmental maps, the first two constrained RDA axes, and the environmental correlation network. Compact source tables are exported with stable English column names.


## 7. Figure 1 与导出数据

Figure 1 合并四个环境变量地图、前两个 constrained RDA axes 和环境相关网络。小型 source tables 使用稳定的英文字段名导出。


In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for axis, variable in zip(axes.flat[:4], focal_variables):
    points = axis.scatter(landrace_indexed["Longitude"], landrace_indexed["Latitude"], c=environment_raw[variable], s=10, cmap="viridis")
    axis.set_title(variable_catalog.set_index("env_id").loc[variable, "env_abbr"])
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    figure.colorbar(points, ax=axis, fraction=0.046)
axes[1, 1].scatter(rda.sample_scores["RDA1"], rda.sample_scores["RDA2"], s=10, color="#2f4b7c")
axes[1, 1].set_xlabel("RDA1")
axes[1, 1].set_ylabel("RDA2")
node_names = nodes["variable_id"].tolist()
angles = np.linspace(0, 2 * np.pi, len(node_names), endpoint=False)
layout = {name: np.array([np.cos(angle), np.sin(angle)]) for name, angle in zip(node_names, angles)}
for edge in edges.itertuples(index=False):
    start, end = layout[edge.source], layout[edge.target]
    color = "#b22222" if edge.correlation > 0 else "#2f4b7c"
    axes[1, 2].plot([start[0], end[0]], [start[1], end[1]], color=color, alpha=min(abs(edge.correlation), 1), linewidth=0.8)
node_xy = np.vstack([layout[name] for name in node_names])
axes[1, 2].scatter(node_xy[:, 0], node_xy[:, 1], s=20 + nodes["degree"].to_numpy() * 8, color="#8aa6c1", edgecolor="black", linewidth=0.3)
for name, position in layout.items():
    axes[1, 2].text(position[0], position[1], name, fontsize=5, ha="center", va="center")
axes[1, 2].set_axis_off()
figure.tight_layout()
save_figure(figure, figure_dir / "figure1.pdf")
landrace_indexed.reset_index().to_csv(supplementary_dataset_dir / "dataset_s1_accessions.tsv", sep="\t", index=False)
variable_catalog.to_csv(supplementary_table_dir / "table_s1_environmental_variables.tsv", sep="\t", index=False)
environment_raw.reset_index().to_csv(figure_data_dir / "environment_variables.tsv", sep="\t", index=False)
rda.sample_scores.to_csv(figure_data_dir / "rda_sample_scores.tsv", sep="\t")
rda.variable_vectors.to_csv(figure_data_dir / "rda_variable_vectors.tsv", sep="\t")
edges.to_csv(figure_data_dir / "environment_correlation_edges.tsv", sep="\t", index=False)
plt.close(figure)


## Manual genotype preparation

Full mode uses the LD-pruned landrace VCF to prepare the genotype matrix and population-structure checkpoint. This explicit manual compute step writes data/checkpoints/generated/environment/population_structure.h5; checkpoint mode reads the identically structured reference file. Striga modelling is not rerun.


## 人工 genotype preparation

Full 模式使用 LD-pruned landrace VCF 准备 genotype matrix 和 population-structure checkpoint。该明确的人工计算步骤写出 data/checkpoints/generated/environment/population_structure.h5；checkpoint 模式读取结构相同的 reference file。项目不重新运行 Striga modelling。


## Saved outputs and next step

Generated files remain in the remote working directory. Inspect the reported row counts, identifiers, quality-control summaries, and figure panels before proceeding to the next numbered notebook. External-tool commands described above remain manual steps.


## 已保存输出与下一步

生成的文件保留在远程工作目录中。进入下一本编号 notebook 前，请检查报告的行数、标识符、quality-control summaries 和 figure panels。以上 external-tool commands 仍需人工执行。
